# 🚁 Notebook 02 — Forces and Gravity

### Where does acceleration actually *come from*?

In Notebook 01 we just **picked** an acceleration number out of thin air. That was fine for
learning the simulation engine, but it's not how the world works. In reality, acceleration is
*produced by* **forces** — pushes and pulls. Gravity pulls the drone down. The motors push it up.

This notebook is where our dot finally becomes a real **drone with mass, fighting gravity**.

---

## 🎯 Learning objectives
- Understand what a **force** is, and what **mass** is, intuitively.
- Use **Newton's second law** (`a = F / m`) — no scary math, just meaning.
- Simulate a drone **falling under gravity** (free fall) and animate it.
- Add **thrust** and compute **net force**.
- Make the drone **hover by hand** — and feel *why* that's so fragile.

## 🧩 Prerequisites
- Notebook 01 (position, velocity, acceleration, Euler integration). We'll recap it in 60 seconds.

## ⏱️ Estimated time
About **45–60 minutes**.

## 🧠 Concepts you know so far
Position · Velocity · Acceleration · Time steps · Euler integration · Simulation · Animation

## 🔜 Concepts you'll learn here
Force · Mass · Newton's second law · Gravity · Free fall · Thrust · Net force · Manual (open-loop) hover


### 🔁 60-second recap of Notebook 01

Our whole simulator is just this loop, repeated in tiny time steps `dt`:

```
new velocity = old velocity + acceleration * dt      # a changes v
new position = old position + new velocity  * dt      # v changes x
```

Last time, `acceleration` was a fixed number we chose. **This time we'll compute it from forces.**
Run the setup cell and let's go.


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
# We only use numpy, matplotlib and ipywidgets in this whole course.
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

# Make animations play as an embedded video/player (works in Google Colab).
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80   # MB, so longer animations don't get cut off
plt.rcParams["figure.figsize"] = (8, 4)      # a comfortable default plot size

# ipywidgets gives us interactive sliders.
from ipywidgets import interact, FloatSlider, IntSlider

# This line helps sliders/widgets show up correctly in Google Colab.
# It does nothing (harmlessly) if you are NOT in Colab.
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready to go!")


## 1. What is a *force*? What is *mass*?

**Force** = a push or a pull. Units: **newtons (N)**. When you push a shopping trolley, catch a
ball, or feel the wind — those are forces. A force is what *makes velocity change*, i.e. what
*causes acceleration*.

**Mass** = how much "stuff" is in an object, i.e. how **hard it is to change its motion**. Units:
**kilograms (kg)**. Intuition:
- Push an empty trolley → it accelerates easily (small mass).
- Push a trolley full of bricks with the *same force* → it barely moves (big mass).

So the same push produces **less** acceleration on a heavier object. That single sentence *is*
the most important law in this notebook.


## 2. Newton's second law — the one equation, explained in words

$$ a = \frac{F}{m} \qquad\text{(acceleration = force divided by mass)}$$

Read it in plain English, three ways — they all say the same thing:
- **More force → more acceleration.** Push harder, speed up quicker. (`a` grows with `F`.)
- **More mass → less acceleration.** Heavier things are sluggish. (`a` shrinks as `m` grows.)
- Equivalently, `F = m × a`: to accelerate a heavy thing quickly, you need a big force.

That's the whole thing. We never need to memorise it — it just *matches your intuition* about
pushing light vs heavy objects. Let's turn it into one tiny Python function.


In [ ]:
def acceleration_from_force(net_force, mass):
    """Newton's second law: acceleration = force / mass.
    net_force in newtons (N), mass in kilograms (kg), returns m/s^2."""
    return net_force / mass

# Sanity checks that match our intuition:
print("Same 10 N force on different masses:")
print(f"  light drone (0.5 kg): a = {acceleration_from_force(10, 0.5):.1f} m/s^2  (accelerates fast)")
print(f"  heavy drone (2.0 kg): a = {acceleration_from_force(10, 2.0):.1f} m/s^2  (accelerates slowly)")
print()
print("Same 1 kg drone, different forces:")
print(f"  gentle push (2 N):  a = {acceleration_from_force(2, 1.0):.1f} m/s^2")
print(f"  strong push (20 N): a = {acceleration_from_force(20, 1.0):.1f} m/s^2")


## 3. Gravity — the force that's *always on*

Earth pulls **everything** downward. Near the ground this pull produces a downward acceleration of
about **g = 9.8 m/s²** — no matter the object's mass (feathers fall as fast as hammers, if there's
no air; this famously got tested on the Moon 🌙).

For a drone of mass `m`, gravity is a downward **force**:

$$ F_{gravity} = m \times g \qquad (\text{pointing down, so we treat it as negative}) $$

We'll use the sign convention: **up is positive (+), down is negative (−).** So gravity is
`-m*g`. Let's drop a drone from 50 m and watch it fall — pure gravity, no motors yet.


In [ ]:
def simulate_falling(mass=1.0, g=9.8, start_height=50.0, dt=0.02, total_time=4.0):
    """Simulate a drone falling under gravity ONLY (motors off).
    Stops the drone at the ground (x = 0). Returns times, positions, velocities."""
    x = start_height     # start high up
    v = 0.0              # released from rest
    t = 0.0
    times, xs, vs = [], [], []
    n_steps = int(total_time / dt)

    for _ in range(n_steps):
        times.append(t); xs.append(x); vs.append(v)

        # The ONLY force is gravity, pointing down (negative):
        gravity_force = -mass * g
        a = acceleration_from_force(gravity_force, mass)   # = -g, as expected

        v = v + a * dt        # Euler: gravity speeds up the downward fall
        x = x + v * dt        # ...which lowers the position

        if x <= 0:            # hit the ground -> stop
            x, v = 0.0, 0.0
            times.append(t + dt); xs.append(x); vs.append(v)
            break
        t = t + dt
    return times, xs, vs

t_fall, x_fall, v_fall = simulate_falling()
print(f"Drone released from 50 m. It hits the ground after about {t_fall[-1]:.2f} s.")
print(f"Notice: acceleration was always -9.8 m/s^2, regardless of the 1 kg mass.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(t_fall, x_fall, color="crimson"); ax1.set_title("Height falls, curving down")
ax1.set_xlabel("time (s)"); ax1.set_ylabel("altitude (m)"); ax1.grid(True, alpha=0.3)
ax2.plot(t_fall, v_fall, color="darkorange"); ax2.set_title("Velocity gets more and more negative")
ax2.set_xlabel("time (s)"); ax2.set_ylabel("velocity (m/s)"); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


## 4. 🎬 Watch the drone fall

Same simulation, now animated so you can *see* gravity accelerate the drone — slow at first, then
faster and faster, until it thumps into the ground.


In [ ]:
t_fall, x_fall, v_fall = simulate_falling(start_height=50.0, total_time=4.0)
frame_ids = np.linspace(0, len(t_fall) - 1, 120).astype(int)

fig, ax = plt.subplots(figsize=(4, 6))
ax.set_xlim(-1, 1); ax.set_ylim(0, 55); ax.set_xticks([])
ax.set_ylabel("altitude (m)"); ax.set_title("Free fall under gravity")
ax.axhline(0, color="saddlebrown", linewidth=3)
drone, = ax.plot([], [], "o", color="crimson", markersize=26)
label = ax.text(-0.9, 50, "", fontsize=11)

def init():
    drone.set_data([], []); label.set_text(""); return drone, label

def update(f):
    i = frame_ids[f]
    drone.set_data([0], [x_fall[i]])
    label.set_text(f"t = {t_fall[i]:.2f} s\nx = {x_fall[i]:.1f} m\nv = {v_fall[i]:.1f} m/s")
    return drone, label

ani = animation.FuncAnimation(fig, update, frames=len(frame_ids), init_func=init,
                              blit=True, interval=40)
plt.close(fig)
HTML(ani.to_jshtml())


## 5. Thrust — the force that fights back

A drone's spinning propellers push air down, and (by reaction) the air pushes the drone **up**.
That upward push is **thrust**, measured in newtons. Unlike gravity, *we* control the thrust — it's
the "knob" the drone commands to fly.

Now the drone feels **two forces at once**:
- Gravity, pulling **down**: `−m·g`
- Thrust, pushing **up**: `+thrust`

What actually decides the motion is the **net force** — everything added together:

$$ F_{net} = \text{thrust} - m\,g $$

Then, as always, `a = F_net / m`, and we Euler-step forward. Three cases to build intuition:

| If… | Net force | Result |
|---|---|---|
| thrust **<** gravity | negative (down) | drone sinks |
| thrust **=** gravity | zero | drone holds still (hover!) |
| thrust **>** gravity | positive (up) | drone climbs |


In [ ]:
def simulate_with_thrust(thrust, mass=1.0, g=9.8, start_height=10.0,
                         dt=0.02, total_time=6.0):
    """Simulate a drone with a FIXED thrust plus gravity. Returns times, positions, velocities."""
    x, v, t = start_height, 0.0, 0.0
    times, xs, vs = [], [], []
    for _ in range(int(total_time / dt)):
        times.append(t); xs.append(x); vs.append(v)

        gravity_force = -mass * g          # down (negative)
        net_force = thrust + gravity_force # thrust is up (positive)
        a = net_force / mass               # Newton's second law

        v = v + a * dt
        x = x + v * dt
        if x < 0:                          # don't fall through the floor
            x, v = 0.0, 0.0
        t = t + dt
    return times, xs, vs

mass, g = 1.0, 9.8
hover_thrust = mass * g       # the "magic" thrust that exactly cancels gravity
print(f"To perfectly balance gravity, thrust must equal m*g = {hover_thrust:.2f} N")

plt.figure(figsize=(8, 4.5))
for thrust, colour, name in [(7.0, "crimson", "too little (7 N) -> sinks"),
                             (hover_thrust, "seagreen", "exact (9.8 N) -> hovers"),
                             (12.0, "royalblue", "too much (12 N) -> climbs")]:
    tL, xL, _ = simulate_with_thrust(thrust)
    plt.plot(tL, xL, color=colour, label=name)
plt.axhline(10, color="gray", linestyle=":", label="start height")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Fixed thrust vs gravity: sink, hover, or climb")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()


## 6. Try to hover by hand 🎛️ — and feel the frustration

Here's the challenge that motivates the *entire rest of the course*. Use the slider to set a fixed
thrust and try to make the drone **hover** (stay near its starting height). The green target line
is where we want it.

Play around, and notice something maddening:
- The "hover" thrust is a razor-thin exact value (`m·g`).
- Just **0.1 N** off and the drone slowly drifts away — up forever, or down to the ground.
- Change the **mass** slider (imagine the drone picks up a package): the old hover thrust is now
  wrong, and it drifts again.

There is **no fixed number** you can set once and trust. That fragility is the problem a
**controller** exists to solve.


In [ ]:
def manual_hover(thrust=9.8, mass=1.0, g=9.8):
    tL, xL, _ = simulate_with_thrust(thrust, mass=mass, g=g,
                                     start_height=10.0, total_time=8.0)
    plt.figure(figsize=(8, 4.5))
    plt.plot(tL, xL, color="mediumvioletred", linewidth=2, label="drone altitude")
    plt.axhline(10, color="seagreen", linestyle="--", label="target (10 m)")
    plt.ylim(0, 25); plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
    needed = mass * g
    verdict = "HOVER!" if abs(thrust - needed) < 0.05 else ("climbing" if thrust > needed else "sinking")
    plt.title(f"thrust={thrust:.2f} N   needs {needed:.2f} N to hover   ->  {verdict}")
    plt.legend(); plt.grid(True, alpha=0.3); plt.show()

interact(manual_hover,
         thrust=FloatSlider(min=6.0, max=14.0, step=0.1, value=9.8),
         mass=FloatSlider(min=0.5, max=2.0, step=0.1, value=1.0),
         g=FloatSlider(min=1.0, max=12.0, step=0.5, value=9.8));


👀 Did you feel it? Getting the drone to *truly* hover requires setting the thrust to a perfect,
invisible value — and any change in mass, gravity, or a puff of wind ruins it. A human (or a fixed
number) simply cannot babysit this. We need something that **watches the altitude and adjusts the
thrust automatically, all the time.** That "something" is **feedback control** — Notebook 03.


## ✅ Summary — what you now understand

- A **force** (newtons) is a push/pull that causes acceleration. **Mass** (kg) is resistance to
  that change.
- **Newton's second law:** `a = F / m`. More force → more acceleration; more mass → less.
- **Gravity** is an always-on downward force `−m·g`, giving acceleration `−g` (≈ −9.8 m/s²)
  regardless of mass.
- **Thrust** is an upward force we control. The motion is decided by the **net force**
  `F_net = thrust − m·g`, then `a = F_net / m`, then the same Euler steps from Notebook 01.
- **Manual hover** requires `thrust = m·g` *exactly*, and is hopelessly fragile — the reason we
  need automatic control.


## ⚠️ Common mistakes
- **Sign errors.** Up is +, down is −. Gravity must be **negative**. A dropped minus sign sends
  your drone flying into space.
- **Forgetting to divide by mass.** Acceleration is `F/m`, not `F`. Forces add up; then you divide
  once by mass.
- **Expecting a fixed thrust to hover forever.** It can't — that's the whole point.
- **Confusing force and acceleration.** Gravity's *force* depends on mass (`m·g`), but the
  *acceleration* it causes (`g`) does not. That's why all objects fall at the same rate.

## 🧭 Mental model
> *"Add up all the forces (with correct + / − signs) to get the net force. Divide by mass to get
> acceleration. Then Euler-step the state forward. Repeat."*

## 🌍 Real-world applications
Rockets balancing thrust against gravity, elevators, cranes, car engines sized to a vehicle's mass,
and every multirotor drone constantly trimming its motor thrust to fight gravity and wind.


## 🧪 Exercises

**Level 1 — Observe.** In the free-fall animation, change `start_height=50.0` to `100.0`. Does the
drone take more or less time to hit the ground? Does it hit *faster* (higher final speed)?

<details><summary>▸ Hint</summary>
Longer to fall means more time for gravity to keep adding downward speed.
</details>
<details><summary>▸ Solution</summary>
More time to fall, and it hits the ground **faster** (larger downward velocity), because gravity
had longer to accelerate it. (You may need to raise <code>total_time</code> so the animation runs
long enough to reach the ground.)
</details>

---
**Level 2 — Modify.** In `simulate_with_thrust`, add a second upward force of **2 N** (pretend a
gust lifts the drone). Recompute the hover thrust needed now. Verify with the slider.

<details><summary>▸ Hint</summary>
Net force becomes <code>thrust + 2 - m*g</code>. Hover now needs a smaller thrust.
</details>
<details><summary>▸ Solution</summary>
With a constant +2 N lift, hover needs <code>thrust = m*g - 2 = 9.8 - 2 = 7.8 N</code>. Any extra
upward force reduces the thrust the motors must supply.
</details>

---
**Level 3 — Predict, then check.** A **2 kg** drone (double the mass) needs how much thrust to
hover under `g = 9.8`? Predict first, then test with the slider by setting mass to 2.0.

<details><summary>▸ Hint</summary>
Hover thrust = m*g.
</details>
<details><summary>▸ Solution</summary>
<code>2 × 9.8 = 19.6 N</code>. Heavier drone → needs proportionally more thrust to hover. (Note the
slider's max is 14 N, so a 2 kg drone can't hover on it — a nice preview of <b>actuator limits</b>
coming in Notebook 09!)
</details>


## ❓ Quick quiz

**Q1.** A feather and a hammer are dropped in a vacuum. Which accelerates faster?
<details><summary>▸ Answer</summary>
**Neither** — both accelerate at g. Gravity's force is bigger on the hammer, but so is its mass,
and the two effects cancel in `a = F/m`.
</details>

**Q2.** Net force on a hovering drone is…
<details><summary>▸ Answer</summary>
**Zero.** Thrust up exactly cancels gravity down, so no acceleration → it holds still.
</details>

**Q3.** You double the thrust while gravity stays the same. The drone…
<details><summary>▸ Answer</summary>
**Accelerates upward** (net force is now strongly positive), climbing faster and faster.
</details>

**Q4.** Why is manual (fixed-thrust) hovering unreliable?
<details><summary>▸ Answer</summary>
The required thrust is an exact value (`m·g`) that changes with mass, gravity, and disturbances.
Any mismatch makes the drone drift, and nothing corrects it. **No feedback.**
</details>


## 🔭 Preview of Notebook 03 — *Open Loop vs Closed Loop*

You just proved that "set a thrust and hope" (**open-loop** control) doesn't work. In Notebook 03
we fix this by giving the drone **eyes**: an **altitude sensor** that measures where it actually
is. We'll compute the **error** (how far it is from the target), and build our first
**feedback loop** — *measure → compare → correct.* You'll see a crude on/off controller wobble
the drone around the target, which sets up the smooth **proportional controller** in Notebook 04.

**Nicely done — you now have a drone with real physics!** 🚁
